# 🚀 Chạy phần mềm Video chuyển đổi Tiếng Việt trên Google Colab (tạo bởi Ngô Thị Hồng Hạnh)

1. **Runtime → Change runtime type → T4 GPU**
2. Chạy lần lượt ô 1 → 4. Mã nguồn **tự tải bản mới nhất từ GitHub**, không cần tải file zip.
3. Nếu đã cài cổng Cloudflare (`GATEWAY_URL`, `GATEWAY_TOKEN` trong Colab Secrets 🔑): **luôn dùng link cố định** `https://video-tieng-viet.<tài-khoản>.workers.dev`, không phải dán link mới.

> Colab tự tắt sau một thời gian không dùng; khi đó chạy lại ô 1 → 4.

## 1. Lấy mã nguồn mới nhất
Tự tải từ GitHub. Nếu không tải được (chưa có mạng tới GitHub), ô này dùng file `xhs2vi.zip` bạn kéo vào thanh 📁 bên trái.

In [ ]:
import os, zipfile, shutil, glob, subprocess
%cd /content
REPO = 'https://github.com/hanhnth2/video-tieng-viet'   # kho mã nguồn (Claude đẩy bản sửa lên đây)
shutil.rmtree('/content/xhs2vi', ignore_errors=True)
r = subprocess.run(['git', 'clone', '--depth', '1', REPO, '/content/xhs2vi'], capture_output=True, text=True)
if r.returncode == 0:
    print('✅ Đã tải bản mới nhất từ GitHub:', subprocess.run(['git', '-C', '/content/xhs2vi', 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())
else:
    print('Không tải được từ GitHub, dùng file zip...')
    zips = sorted(glob.glob('/content/xhs2vi*.zip'), key=os.path.getmtime)
    if zips:
        zpath = zips[-1]
    else:
        from google.colab import files
        up = files.upload()
        zpath = '/content/' + list(up.keys())[0]
    print('Dùng file:', zpath)
    zipfile.ZipFile(zpath).extractall('/content')
%cd /content/xhs2vi

## 2. Cài đặt (3–5 phút)

In [ ]:
!pip -q install -r requirements.txt
# Thư viện CUDA 12 cho nhận dạng giọng trên GPU (sửa lỗi 'libcublas.so.12 is not found')
!pip -q install nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
!python -c "import rapidocr, faster_whisper, edge_tts; print('Cài đặt OK')"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo '⚠ Chưa bật GPU: Runtime → Change runtime type → T4 GPU'
!apt-get -qq install -y fonts-noto-core > /dev/null
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared

## 3. Cấu hình (tuỳ chọn)
Đặt trong **Colab Secrets** (biểu tượng 🔑 bên trái, bật *Notebook access*):
- `GATEWAY_URL` + `GATEWAY_TOKEN`: cổng link cố định trên Cloudflare (xem `HUONG-DAN-CLOUDFLARE.md`)
- `XHS_COOKIE`, `ANTHROPIC_API_KEY`, `FPT_API_KEY`, `APP_PASSWORD`

In [ ]:
import os
import shutil
if shutil.which('nvidia-smi'):
    os.environ['WHISPER_MODEL'] = 'large-v3'   # có GPU: mô hình lớn, chính xác nhất
    print('✅ Có GPU – dùng mô hình large-v3')
else:
    os.environ['WHISPER_MODEL'] = 'small'      # không có GPU: mô hình nhỏ cho nhanh
    print('⚠ Không có GPU – chạy bằng CPU với mô hình small (chậm hơn, kém chính xác hơn một chút)')
os.environ['DATA_DIR'] = '/content/data'
try:
    from google.colab import userdata
    for k in ('ANTHROPIC_API_KEY', 'FPT_API_KEY', 'XHS_COOKIE', 'APP_PASSWORD', 'GATEWAY_URL', 'GATEWAY_TOKEN'):
        try:
            v = userdata.get(k)
            if v: os.environ[k] = v.strip(); print('Đã nạp', k)
        except Exception: pass
except ImportError:
    pass

## 4. Chạy web

In [ ]:
import subprocess, time, re, urllib.request, os

def cuda_lib_path():
    """Đường dẫn thư viện CUDA 12 cài từ pip, để faster-whisper tìm thấy libcublas.so.12 / libcudnn."""
    import glob, site
    dirs = []
    for sp in site.getsitepackages():
        dirs += glob.glob(sp + '/nvidia/*/lib')
    return ':'.join(dirs + [os.environ.get('LD_LIBRARY_PATH', '')])

def start_server():
    subprocess.run("pkill -f 'uvicorn app.main' ; true", shell=True)
    env = dict(os.environ, LD_LIBRARY_PATH=cuda_lib_path())
    subprocess.Popen(['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'], env=env,
                     stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT, cwd='/content/xhs2vi')
    for _ in range(60):
        try:
            urllib.request.urlopen('http://localhost:8000/api/health', timeout=2); return True
        except Exception:
            time.sleep(1)
    return False

def start_tunnel():
    subprocess.run("pkill -f cloudflared ; true", shell=True)
    log = '/content/cloudflared.log'
    open(log, 'w').close()
    # ghi nhật ký ra file (không dùng PIPE để cloudflared không bị treo)
    subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                     stdout=open(log, 'a'), stderr=subprocess.STDOUT)
    url = None
    for _ in range(90):
        txt = open(log).read()
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', txt)
        if m: url = m.group(0)
        if url and 'Registered tunnel connection' in txt:
            return url
        time.sleep(1)
    return url

def register(url):
    """Báo link tạm cho cổng Cloudflare -> web/app dùng link cố định."""
    gw, tok = os.environ.get('GATEWAY_URL', '').rstrip('/'), os.environ.get('GATEWAY_TOKEN', '')
    if not (gw and tok and url):
        return None
    import json
    req = urllib.request.Request(gw + '/__register', data=json.dumps({'url': url, 'token': tok}).encode(),
                                 headers={'content-type': 'application/json',
                                          # Cloudflare chặn (lỗi 1010) yêu cầu tự xưng 'Python-urllib' -> giới thiệu như trình duyệt
                                          'user-agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36 VideoVietColab/1.0'},
                                 method='POST')
    for _ in range(3):
        try:
            res = json.loads(urllib.request.urlopen(req, timeout=20).read())
            if res.get('ok'):
                return gw
            print('⚠ Cổng báo lỗi:', res.get('error'))
            return None
        except urllib.error.HTTPError as e:
            print('⚠ Cổng báo lỗi:', e.read().decode()[:200]); return None
        except Exception as e:
            print('… thử lại kết nối cổng:', e); time.sleep(3)
    return None

print('Khởi động web...', 'OK' if start_server() else 'LỖI – xem ô log bên dưới')
url = start_tunnel()
print('Chờ thêm vài giây để link hoạt động...'); time.sleep(8)
gw = register(url)
if gw:
    print('\n✅ Đã kết nối cổng. LINK CỐ ĐỊNH (dùng mãi):', gw)
else:
    print('\n👉 Mở web tại:', url)
    print('   (Link tạm, đổi mỗi lần chạy. Cài cổng Cloudflare để có link cố định.)')

## 5. Khi link báo lỗi 1033 / không vào được
Chạy ô này để tạo **link mới** (web vẫn giữ nguyên lịch sử video).

In [ ]:
try:
    urllib.request.urlopen('http://localhost:8000/api/health', timeout=3); print('Web vẫn chạy')
except Exception:
    print('Web đã dừng, khởi động lại...', start_server())
url = start_tunnel(); time.sleep(8)
gw = register(url)
print('✅ Link cố định vẫn dùng được:', gw) if gw else print('👉 Link mới:', url)

## 6. Giữ Colab không bị ngắt
Chạy ô này và **để tab Colab mở** khi đang xử lý video. Bấm ■ để dừng khi xong.

In [ ]:
import time
while True:
    try:
        urllib.request.urlopen('http://localhost:8000/api/health', timeout=3); ok = 'web OK'
    except Exception:
        ok = 'web DỪNG'
    print(time.strftime('%H:%M:%S'), ok, end='\r')
    time.sleep(60)

In [ ]:
# Xem log khi cần gỡ lỗi
!tail -n 30 /content/server.log
!echo ----- cloudflared -----
!tail -n 15 /content/cloudflared.log